# MultiDiffusion SD1.5 + Hyper-SD Profiled Run on Kaggle

This notebook runs the experiment row:

```text
Method    : MultiDiffusion (MD)
Model     : Stable Diffusion 1.5
Checkpoint: runwayml/stable-diffusion-v1-5
Sampler   : Hyper-SD 4-step via DDIMScheduler(timestep_spacing="trailing")
Accel     : ByteDance/Hyper-SD/Hyper-SD15-4steps-lora.safetensors
Resolution: 512x512
Manifest  : depends on RUN_PROFILE
Samples   : depends on RUN_PROFILE
Metrics   : FID, IS, CLIP(fg), CLIP(bg), Time(s)
```

Goal: validate with smoke/mini manifests or benchmark full **MultiDiffusion + Hyper-SD** on the same 1073 COCO samples used for SemanticDraw SD1.5 + Hyper-SD.

Important notes:

- Original MultiDiffusion treats background as region 0, so the input must be `prompts = [background_prompt] + foreground_prompts` and `masks = [background_mask] + foreground_masks`.
- `background_mask = 1 - union(foreground_masks)`.
- The region/window fusion follows `Baseline/MultiDiffusion-master/MultiDiffusion-master/region_based.py`: each region/window is denoised separately, then blended by `value / count` using masks.
- Intentional sampler change: original DDIM 50-step is replaced by Hyper-SD SD1.5 4-step, using `DDIMScheduler(timestep_spacing="trailing")` and LoRA `ByteDance/Hyper-SD/Hyper-SD15-4steps-lora.safetensors`.
- This notebook does **not** use SemanticDraw improvements such as mask-centering, latent pre-averaging, bootstrap leak removal, or noise-level mask quantization.

To switch between smoke test and full benchmark, edit `RUN_PROFILE` in the config cell:

```python
RUN_PROFILE = "smoke"    # quick validation
RUN_PROFILE = "full1073" # official benchmark
```


## 0. Yêu cầu Kaggle

Trước khi Run All:

- Bật `Internet = On`.
- Bật `Accelerator = GPU`.
- Nếu checkpoint/Hugging Face bị yêu cầu quyền, thêm Kaggle Secret `HF_TOKEN`.
- Không cài lại `torch`; notebook chỉ cài thư viện bổ sung.


In [ ]:
# Cài các thư viện cần thiết cho generation + metric evaluation.
# Không cài lại torch để tránh làm lệch môi trường GPU mặc định của Kaggle.
# Quan trọng: gỡ torchao. Một số Kaggle image có torchao==0.10.0;
# peft mới thấy torchao nhưng yêu cầu >0.16.0, gây lỗi khi load Hyper-SD LoRA.
import sys
import subprocess

packages = [
    "diffusers>=0.30.0",
    "transformers>=4.44.0",
    "accelerate",
    "peft",
    "huggingface_hub",
    "safetensors",
    "sentencepiece",
    "protobuf",
    "einops",
    "pycocotools",
    "matplotlib",
    "tqdm",
    "pandas>=2.0",
    "open-clip-torch>=2.24.0",
    "torch-fidelity>=0.3.0",
    "torchmetrics>=1.4",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
print("[OK] Dependencies are ready. torchao is removed to avoid PEFT LoRA compatibility errors.")


In [ ]:
# Clone repo nếu notebook chưa nằm trong repo clone.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/GOx9-P/AnchorDraw.git"
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()


def is_repo_root(path: Path) -> bool:
    # Notebook chạy bằng code trong Ours/src. Folder baseline MultiDiffusion chỉ dùng để đối chiếu,
    # nên không bắt nó tồn tại khi clone từ GitHub/Kaggle.
    return (
        (path / "Ours" / "src" / "data").exists()
        and (path / "Ours" / "src" / "baselines").exists()
        and (path / "Ours" / "data_manifests").exists()
    )


def find_repo_root() -> Path | None:
    starts = [
        Path.cwd(),
        Path.cwd() / "AnchorDraw",
        WORK_DIR / "AnchorDraw",
        WORK_DIR / "AnchorDraw" / "AnchorDraw",
        WORK_DIR / "anchor_draw",
    ]
    checked = set()
    for start in starts:
        for path in [start, *start.parents]:
            path = path.resolve()
            if path in checked:
                continue
            checked.add(path)
            if is_repo_root(path):
                return path
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    clone_target = WORK_DIR / "AnchorDraw"
    if not clone_target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)], check=True)
    REPO_ROOT = find_repo_root()

assert REPO_ROOT is not None and is_repo_root(REPO_ROOT), "Không tìm thấy repo root sau khi clone."
print(f"[OK] Repo root: {REPO_ROOT}")


In [ ]:
# Configure run profile for MultiDiffusion SD1.5 + Hyper-SD.
# Change only this variable to switch between smoke/mini/full.
# Options: "smoke", "mini32", "mini128", "full1073".
RUN_PROFILE = "full1073"

PROFILE_CONFIGS = {
    "smoke": {
        "manifest": REPO_ROOT / "Ours" / "test_sets" / "manifests" / "smoke" / "coco_val2017_multidiffusion_coco_all_512x512_smoke_bs8.jsonl",
        "expected_samples": 8,
        "label": "smoke_bs8",
        "max_display_results": 8,
    },
    "mini32": {
        "manifest": REPO_ROOT / "Ours" / "test_sets" / "manifests" / "mini32" / "coco_val2017_multidiffusion_coco_all_512x512_mini32.jsonl",
        "expected_samples": 32,
        "label": "mini32",
        "max_display_results": 8,
    },
    "mini128": {
        "manifest": REPO_ROOT / "Ours" / "test_sets" / "manifests" / "mini128" / "coco_val2017_multidiffusion_coco_all_512x512_mini128.jsonl",
        "expected_samples": 128,
        "label": "mini128",
        "max_display_results": 8,
    },
    "full1073": {
        "manifest": REPO_ROOT / "Ours" / "data_manifests" / "coco_val2017_multidiffusion_coco_all_512x512_all.jsonl",
        "expected_samples": 1073,
        "label": "full1073",
        "max_display_results": 8,
    },
}

assert RUN_PROFILE in PROFILE_CONFIGS, f"Unknown RUN_PROFILE={RUN_PROFILE!r}. Choose one of {tuple(PROFILE_CONFIGS)}"
RUN_CONFIG = PROFILE_CONFIGS[RUN_PROFILE]
RUN_MANIFEST = RUN_CONFIG["manifest"]
RUN_LABEL = RUN_CONFIG["label"]
EXPECTED_SAMPLES = RUN_CONFIG["expected_samples"]

COCO_ROOT = Path(os.environ.get("COCO_ROOT", "/kaggle/working/COCO"))
OUTPUT_DIR = Path(f"/kaggle/working/multidiffusion_sd15_hypersd_{RUN_LABEL}_outputs")
MASK_CACHE_DIR = Path("/kaggle/working/multidiffusion_sd15_hypersd_mask_cache")
METRICS_OUTPUT_DIR = Path(f"/kaggle/working/multidiffusion_sd15_hypersd_{RUN_LABEL}_metrics")
EXPERIMENT_ID = f"multidiffusion_sd15_hypersd_{RUN_LABEL}"

MODEL_ID = "runwayml/stable-diffusion-v1-5"
HYPER_SD_REPO_ID = "ByteDance/Hyper-SD"
HYPER_SD_WEIGHT_NAME = "Hyper-SD15-4steps-lora.safetensors"
HYPER_SD_NUM_INFERENCE_STEPS = 4
HYPER_SD_LORA_SCALE = 1.0

# Diffusers Hyper-SD usually uses guidance_scale=0 to disable CFG.
# In this custom MultiDiffusion loop we still compute uncond + scale * (cond - uncond),
# so scale=1.0 corresponds to conditional-only/no-CFG.
HYPER_SD_GUIDANCE_SCALE = 1.0

TARGET_SIZE = (512, 512)
BATCH_SIZE = 8
BASE_SEED = 2024

# Original MultiDiffusion uses random-background bootstrapping.
# With Hyper-SD 4-step, keep 1 bootstrap step to match the fast SemanticDraw SD1.5 spirit.
BOOTSTRAPPING = 1

NEGATIVE_PROMPT = ""
MAX_DISPLAY_RESULTS = RUN_CONFIG["max_display_results"]

METRIC_NAMES = ("fid", "is", "clip_fg", "clip_bg", "time")
METRIC_BATCH_SIZE = 8
CLIP_BATCH_SIZE = 16
IS_SPLITS = 10
METRICS_REPORT_PREFIX = f"{EXPERIMENT_ID}_metrics"

assert RUN_MANIFEST.exists(), f"Missing manifest: {RUN_MANIFEST}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Run profile: {RUN_PROFILE} ({RUN_LABEL})")
print(f"[OK] Manifest: {RUN_MANIFEST}")
print(f"[OK] Expected samples: {EXPECTED_SAMPLES}")
print(f"[OK] COCO root: {COCO_ROOT}")
print(f"[OK] Output dir: {OUTPUT_DIR}")
print(f"[OK] Metrics output dir: {METRICS_OUTPUT_DIR}")
print(f"[OK] Batch size: {BATCH_SIZE}")
print(f"[OK] Experiment ID: {EXPERIMENT_ID}")
print(f"[OK] Hyper-SD LoRA: {HYPER_SD_REPO_ID}/{HYPER_SD_WEIGHT_NAME}")


In [ ]:
# Tải COCO val2017 nếu Kaggle runtime chưa có sẵn dữ liệu.
# Lưu ý: trên một số Kaggle runtime, HTTPS của images.cocodataset.org có thể lỗi SSL.
# Vì vậy cell này ưu tiên HTTP official COCO và có nhiều fallback download.
import ssl
import urllib.request
import zipfile

COCO_ROOT.mkdir(parents=True, exist_ok=True)

VAL_ZIP_URLS = [
    "http://images.cocodataset.org/zips/val2017.zip",
    "https://images.cocodataset.org/zips/val2017.zip",
]
ANN_ZIP_URLS = [
    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
]
val_zip = COCO_ROOT / "val2017.zip"
ann_zip = COCO_ROOT / "annotations_trainval2017.zip"


def run_download_command(cmd: list[str]) -> bool:
    try:
        subprocess.run(cmd, check=True)
        return True
    except Exception as exc:
        print(f"[WARN] Download command failed: {' '.join(cmd[:2])} -> {exc}")
        return False


def download_file(urls: list[str], dst: Path) -> None:
    if dst.exists() and dst.stat().st_size > 0:
        print(f"[SKIP] Already downloaded: {dst.name}")
        return

    last_error = None
    for url in urls:
        print(f"[DOWNLOAD] {url}")

        if run_download_command(["wget", "-c", "--no-check-certificate", "-O", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                return

        if run_download_command(["curl", "-L", "-k", "--retry", "3", "-o", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                return

        try:
            context = ssl._create_unverified_context()
            with urllib.request.urlopen(url, context=context, timeout=120) as response:
                with dst.open("wb") as f:
                    f.write(response.read())
            if dst.exists() and dst.stat().st_size > 0:
                return
        except Exception as exc:
            last_error = exc
            print(f"[WARN] urllib failed for {url}: {exc}")

    raise RuntimeError(
        f"Cannot download {dst.name}. Last error: {last_error}. "
        "Check Kaggle Internet setting, or attach COCO val2017 as a Kaggle Dataset and set COCO_ROOT."
    )


def unzip_if_missing(zip_path: Path, marker_path: Path) -> None:
    if marker_path.exists():
        print(f"[SKIP] Already extracted: {marker_path}")
        return
    print(f"[UNZIP] {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(COCO_ROOT)


download_file(VAL_ZIP_URLS, val_zip)
download_file(ANN_ZIP_URLS, ann_zip)
unzip_if_missing(val_zip, COCO_ROOT / "val2017" / "000000000139.jpg")
unzip_if_missing(ann_zip, COCO_ROOT / "annotations" / "instances_val2017.json")

assert (COCO_ROOT / "val2017").exists(), "Missing COCO val2017 images."
assert (COCO_ROOT / "annotations" / "instances_val2017.json").exists(), "Missing instances_val2017.json."
assert (COCO_ROOT / "annotations" / "captions_val2017.json").exists(), "Missing captions_val2017.json."
print("[OK] COCO val2017 is ready.")


In [ ]:
# Import Ours dataloader, metric package, and MultiDiffusion Hyper-SD wrapper.
import hashlib
import importlib
import importlib.util
import json
import math
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from PIL import Image

OURS_SRC = REPO_ROOT / "Ours" / "src"
BASELINE_REGION_FILE = REPO_ROOT / "Baseline" / "MultiDiffusion-master" / "MultiDiffusion-master" / "region_based.py"

# Put Ours/src first so data/metrics/baselines resolve to the intended package.
sys.path = [str(OURS_SRC)] + [p for p in sys.path if p != str(OURS_SRC)]

from baselines import MultiDiffusionHyperSD
from data import COCORegionConfig, batch_item_to_semanticdraw_inputs, build_coco_region_dataloader
from data.visualize import make_mask_overlay

print("[OK] Imports are ready.")
if BASELINE_REGION_FILE.exists():
    region_sha256 = hashlib.sha256(BASELINE_REGION_FILE.read_bytes()).hexdigest()
    print("[OK] Baseline MultiDiffusion region file:", BASELINE_REGION_FILE)
    print("[OK] Baseline region_based.py SHA256:", region_sha256)
else:
    print("[WARN] Baseline MultiDiffusion region_based.py not found in this clone; running Ours/src/baselines implementation.")


In [ ]:
# Tạo dataloader theo RUN_PROFILE đã chọn.
config = COCORegionConfig(
    coco_root=COCO_ROOT,
    split="val2017",
    instances_json=COCO_ROOT / "annotations" / "instances_val2017.json",
    captions_json=COCO_ROOT / "annotations" / "captions_val2017.json",
    manifest_path=RUN_MANIFEST,
    profile="multidiffusion_coco_all",
    model_family="sd15",
    target_size=TARGET_SIZE,
    return_image=True,
    cache_resized_masks=True,
    cache_dir=MASK_CACHE_DIR,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)

loader = build_coco_region_dataloader(config, shuffle=False, drop_last=False)
dataset_size = len(loader.dataset)
num_batches = len(loader)
if EXPECTED_SAMPLES is not None:
    assert dataset_size == EXPECTED_SAMPLES, f"Profile {RUN_PROFILE} expected {EXPECTED_SAMPLES} samples, got {dataset_size}."
preview_batch = next(iter(loader))

print(f"[OK] Manifest records: {dataset_size}")
print(f"[OK] Dataloader batches: {num_batches} batch(es) x up to {BATCH_SIZE} sample(s)")
print(f"[OK] First batch size: {len(preview_batch['sample_ids'])}")
print(f"[OK] First batch masks shape: {tuple(preview_batch['masks'].shape)}  # (B, Pmax, C, H, W)")
print("First batch sample IDs:")
for sample_id in preview_batch["sample_ids"]:
    print(" -", sample_id)


## 1. Chuẩn hóa Input Cho MultiDiffusion

SemanticDraw nhận foreground masks riêng. MultiDiffusion thì cần `background_mask` nằm trong danh sách mask. Cell dưới tạo payload theo đúng format:

```text
prompts = [background_prompt, fg_prompt_1, fg_prompt_2, ...]
masks   = [background_mask, fg_mask_1, fg_mask_2, ...]
```


In [ ]:
def md_escape(text: object) -> str:
    return str(text).replace("\n", " ").replace("|", "\\|")


def seed_everything(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def image_stats(image: Image.Image) -> dict:
    import numpy as np

    arr = torch.from_numpy(np.array(image.convert("RGB")))
    return {
        "min": int(arr.min().item()),
        "max": int(arr.max().item()),
        "mean": float(arr.float().mean().item()),
        "std": float(arr.float().std().item()),
    }


def make_multidiffusion_payload(batch: dict, index: int) -> dict:
    item = batch_item_to_semanticdraw_inputs(batch, index)
    fg_masks = item["masks"].float().cpu()
    fg_union = fg_masks.sum(dim=0, keepdim=True).clamp(0, 1)
    background_mask = (1.0 - fg_union).clamp(0, 1)
    all_masks = torch.cat([background_mask, fg_masks], dim=0)

    prompts = [item["background_prompt"], *item["prompts"]]
    negative_prompts = [NEGATIVE_PROMPT for _ in prompts]
    metadata = item["metadata"]

    return {
        "sample_id": metadata["sample_id"],
        "image_id": metadata["image_id"],
        "file_name": metadata["file_name"],
        "height": item["height"],
        "width": item["width"],
        "prompts": prompts,
        "negative_prompts": negative_prompts,
        "foreground_prompts": item["prompts"],
        "category_names": metadata["category_names"],
        "annotation_ids": metadata["annotation_ids"],
        "area_ratios": metadata["area_ratios"],
        "foreground_masks": fg_masks,
        "all_masks": all_masks,
        "metadata": metadata,
    }


def display_multidiffusion_result(payload: dict, original: Image.Image, overlay: Image.Image, generated: Image.Image, elapsed: float, generated_path: Path) -> None:
    rows = ["| Region | Prompt | Annotation | Area ratio |", "|---|---|---:|---:|"]
    rows.append(f"| Background | {md_escape(payload['prompts'][0])} | - | - |")
    for label, prompt, ann_id, area in zip(payload["category_names"], payload["foreground_prompts"], payload["annotation_ids"], payload["area_ratios"]):
        rows.append(f"| {md_escape(label)} | {md_escape(prompt)} | {ann_id} | {float(area):.4f} |")

    display(Markdown(
        f"### `{payload['sample_id']}`\n"
        f"- image_id: `{payload['image_id']}`\n"
        f"- file: `{payload['file_name']}`\n"
        f"- prompt/mask count: `{len(payload['prompts'])}`\n"
        f"- generated path: `{generated_path}`\n"
        f"- elapsed: `{elapsed:.2f}s`\n"
        f"- generated stats: `{image_stats(generated)}`\n\n"
        + "\n".join(rows)
    ))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original)
    axes[0].set_title("COCO original resized")
    axes[1].imshow(overlay)
    axes[1].set_title("Foreground mask overlay")
    axes[2].imshow(generated)
    axes[2].set_title("MultiDiffusion + Hyper-SD generated")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("[OK] Helper functions are ready.")


In [ ]:
# Login Hugging Face nếu có token trong Kaggle Secret hoặc biến môi trường.
def maybe_login_to_huggingface() -> None:
    token = os.environ.get("HF_TOKEN")
    if token is None:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token:
        from huggingface_hub import login
        login(token=token)
        print("[OK] Hugging Face token loaded.")
    else:
        print("[INFO] No HF_TOKEN found. Public/gated model access depends on your Hugging Face permissions.")


assert torch.cuda.is_available(), "Kaggle runtime chưa bật GPU. Hãy bật Accelerator = GPU rồi chạy lại."
device = torch.device("cuda:0")
dtype = torch.float16

maybe_login_to_huggingface()
print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# Guard against torchao/PEFT issues before loading Hyper-SD LoRA.
# If torchao was just imported in the load-pipeline cell, restart the Kaggle session and Run All.
importlib.invalidate_caches()

if "torchao" in sys.modules:
    raise RuntimeError(
        "torchao is already imported in this Python session. Restart the Kaggle session, run the dependency cell, then Run All. "
        "MultiDiffusion SD1.5 + Hyper-SD LoRA does not need torchao."
    )

if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError(
        "torchao is still installed/importable in this runtime. Run the dependency cell, then restart the Kaggle session and Run All. "
        "MultiDiffusion SD1.5 + Hyper-SD LoRA does not need torchao."
    )

print("[OK] torchao is not importable; PEFT should skip torchao LoRA dispatch.")


In [ ]:
# Load MultiDiffusion + Hyper-SD model.
seed_everything(BASE_SEED)
md_hyper = MultiDiffusionHyperSD(
    model_id=MODEL_ID,
    hyper_sd_repo_id=HYPER_SD_REPO_ID,
    hyper_sd_weight_name=HYPER_SD_WEIGHT_NAME,
    device=device,
    dtype=dtype,
    num_inference_steps=HYPER_SD_NUM_INFERENCE_STEPS,
    lora_scale=HYPER_SD_LORA_SCALE,
)

print("Scheduler:", type(md_hyper.scheduler).__name__)
print("Timestep spacing:", getattr(md_hyper.scheduler.config, "timestep_spacing", None))
print("Model:", MODEL_ID)
print("Hyper-SD LoRA:", f"{HYPER_SD_REPO_ID}/{HYPER_SD_WEIGHT_NAME}")
print("Inference steps:", HYPER_SD_NUM_INFERENCE_STEPS)
print("timesteps:", [int(t.item()) for t in md_hyper.timesteps])
print("guidance_scale:", HYPER_SD_GUIDANCE_SCALE)
assert type(md_hyper.scheduler).__name__ == "DDIMScheduler", "Expected DDIMScheduler for Hyper-SD."
assert getattr(md_hyper.scheduler.config, "timestep_spacing", None) == "trailing"


In [ ]:
# Sanity check: one simple prompt with a full mask to test checkpoint/scheduler/LoRA/VAE.
seed_everything(BASE_SEED)
sanity_mask = torch.ones(1, 1, TARGET_SIZE[0], TARGET_SIZE[1])
sanity_image = md_hyper.generate(
    masks=sanity_mask,
    prompts=["a studio photo of a teddy bear on a clean table"],
    negative_prompts=[NEGATIVE_PROMPT],
    height=TARGET_SIZE[0],
    width=TARGET_SIZE[1],
    guidance_scale=HYPER_SD_GUIDANCE_SCALE,
    bootstrapping=0,
    num_inference_steps=HYPER_SD_NUM_INFERENCE_STEPS,
)
print("[SANITY] image stats:", image_stats(sanity_image))
display(sanity_image.resize((384, 384)))


In [ ]:
# Run generation for every sample in the manifest and display results.
summary = []
global_index = 0

for batch_index, batch in enumerate(loader):
    print(f"[BATCH] {batch_index + 1}/{len(loader)} - {len(batch['sample_ids'])} sample(s)")

    for local_index, sample_id in enumerate(batch["sample_ids"]):
        payload = make_multidiffusion_payload(batch, local_index)
        original = batch["images"][local_index].resize((payload["width"], payload["height"]), Image.Resampling.BILINEAR)
        overlay = make_mask_overlay(original, payload["foreground_masks"], payload["category_names"], alpha=0.45)

        seed = BASE_SEED + global_index
        seed_everything(seed)

        tic = time.perf_counter()
        generated = md_hyper.generate(
            masks=payload["all_masks"],
            prompts=payload["prompts"],
            negative_prompts=payload["negative_prompts"],
            height=payload["height"],
            width=payload["width"],
            guidance_scale=HYPER_SD_GUIDANCE_SCALE,
            bootstrapping=BOOTSTRAPPING,
            num_inference_steps=HYPER_SD_NUM_INFERENCE_STEPS,
        )
        elapsed = time.perf_counter() - tic

        stem = f"{global_index:04d}_{payload['sample_id']}"
        generated_path = OUTPUT_DIR / f"{stem}_generated.png"
        overlay_path = OUTPUT_DIR / f"{stem}_overlay.png"
        generated.save(generated_path)
        overlay.save(overlay_path)

        with Image.open(generated_path) as check_img:
            check_img.verify()

        summary.append({
            "index": global_index,
            "batch_index": batch_index,
            "local_index": local_index,
            "sample_id": payload["sample_id"],
            "image_id": payload["image_id"],
            "file_name": payload["file_name"],
            "seed": seed,
            "method": "multidiffusion",
            "model_family": "sd15",
            "model_id": MODEL_ID,
            "sampler": "hyper-sd",
            "scheduler": type(md_hyper.scheduler).__name__,
            "timestep_spacing": getattr(md_hyper.scheduler.config, "timestep_spacing", None),
            "hyper_sd_repo": HYPER_SD_REPO_ID,
            "hyper_sd_weight": HYPER_SD_WEIGHT_NAME,
            "num_inference_steps": HYPER_SD_NUM_INFERENCE_STEPS,
            "guidance_scale": HYPER_SD_GUIDANCE_SCALE,
            "bootstrapping": BOOTSTRAPPING,
            "num_regions_including_background": len(payload["prompts"]),
            "elapsed_sec": elapsed,
            "generated_path": str(generated_path),
            "overlay_path": str(overlay_path),
        })

        should_display = MAX_DISPLAY_RESULTS is None or global_index < MAX_DISPLAY_RESULTS
        if should_display:
            display_multidiffusion_result(payload, original, overlay, generated, elapsed, generated_path)

        global_index += 1
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

summary_path = OUTPUT_DIR / "generation_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

display(Markdown(
    f"## Done\n"
    f"Generated `{len(summary)}` image(s) from `{dataset_size}` manifest record(s). "
    f"Summary saved to `{summary_path}`."
))
summary[:5]


In [ ]:
# Export ảnh đã sinh sang folder chuẩn để đo metric/reproduce sau này và nén thành zip.
import csv
import shutil
import zipfile

METRIC_EXPORT_EXPERIMENT_ID = EXPERIMENT_ID
METRIC_EXPORT_ROOT = Path("/kaggle/working/anchordraw_metric_exports")
METRIC_EXPORT_DIR = METRIC_EXPORT_ROOT / METRIC_EXPORT_EXPERIMENT_ID
METRIC_EXPORT_GENERATED_DIR = METRIC_EXPORT_DIR / "generated_images"
METRIC_EXPORT_ORIGINAL_DIR = METRIC_EXPORT_DIR / "original_images"
METRIC_EXPORT_MANIFEST_JSONL = METRIC_EXPORT_DIR / "metric_generated_manifest.jsonl"
METRIC_EXPORT_MANIFEST_CSV = METRIC_EXPORT_DIR / "metric_generated_manifest.csv"
METRIC_EXPORT_SUMMARY_JSON = METRIC_EXPORT_DIR / "export_summary.json"

COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT = False

for path in [METRIC_EXPORT_DIR, METRIC_EXPORT_GENERATED_DIR]:
    path.mkdir(parents=True, exist_ok=True)
if COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT:
    METRIC_EXPORT_ORIGINAL_DIR.mkdir(parents=True, exist_ok=True)


def _safe_name(text: object, max_len: int = 120) -> str:
    keep = []
    for ch in str(text):
        keep.append(ch if ch.isalnum() or ch in ("-", "_", ".") else "_")
    name = "".join(keep).strip("_")
    return name[:max_len] or "sample"


def _load_manifest_records_by_sample_id(manifest_path: Path) -> dict:
    records = {}
    with manifest_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                records[record["sample_id"]] = record
    return records


if "summary" not in globals() or not summary:
    with (OUTPUT_DIR / "generation_summary.json").open("r", encoding="utf-8") as f:
        summary = json.load(f)

manifest_by_sample_id = _load_manifest_records_by_sample_id(Path(RUN_MANIFEST))
metric_records = []

for row_position, gen in enumerate(summary):
    sample_id = gen.get("sample_id")
    manifest_record = manifest_by_sample_id.get(sample_id, {})
    image_id = int(gen.get("image_id", manifest_record.get("image_id", -1)))
    file_name = gen.get("file_name", manifest_record.get("file_name"))
    source_generated_path = Path(gen["generated_path"])
    if not source_generated_path.exists():
        raise FileNotFoundError(source_generated_path)
    with Image.open(source_generated_path) as check_img:
        check_img.verify()

    metric_index = int(gen.get("index", row_position))
    canonical_name = f"{metric_index:06d}__coco_{image_id:012d}__{_safe_name(sample_id)}__generated.png"
    metric_generated_path = METRIC_EXPORT_GENERATED_DIR / canonical_name
    if source_generated_path.resolve() != metric_generated_path.resolve():
        shutil.copy2(source_generated_path, metric_generated_path)

    coco_original_path = Path(COCO_ROOT) / "val2017" / file_name if file_name else None
    copied_original_path = None
    if COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT and coco_original_path is not None and coco_original_path.exists():
        original_name = f"{metric_index:06d}__coco_{image_id:012d}__{_safe_name(sample_id)}__original.jpg"
        copied_original_path = METRIC_EXPORT_ORIGINAL_DIR / original_name
        shutil.copy2(coco_original_path, copied_original_path)

    metric_records.append({
        "metric_index": metric_index,
        "experiment_id": METRIC_EXPORT_EXPERIMENT_ID,
        "sample_id": sample_id,
        "image_id": image_id,
        "file_name": file_name,
        "generated_image_path": str(metric_generated_path),
        "generated_image_relative_path": str(metric_generated_path.relative_to(METRIC_EXPORT_DIR)),
        "source_generated_path": str(source_generated_path),
        "coco_original_path": str(coco_original_path) if coco_original_path is not None else None,
        "copied_original_path": str(copied_original_path) if copied_original_path is not None else None,
        "source_manifest_path": str(RUN_MANIFEST),
        "source_output_dir": str(OUTPUT_DIR),
        "background_prompt": manifest_record.get("caption"),
        "foreground_prompts": manifest_record.get("foreground_prompts"),
        "category_names": manifest_record.get("category_names"),
        "category_ids": manifest_record.get("category_ids"),
        "annotation_ids": manifest_record.get("annotation_ids"),
        "area_ratios": manifest_record.get("area_ratios"),
        "target_size": manifest_record.get("target_size"),
        "original_size": manifest_record.get("original_size"),
        "method": gen.get("method"),
        "model_family": gen.get("model_family"),
        "model_id": gen.get("model_id"),
        "sampler": gen.get("sampler"),
        "scheduler": gen.get("scheduler"),
        "hyper_sd_repo": gen.get("hyper_sd_repo"),
        "hyper_sd_weight": gen.get("hyper_sd_weight"),
        "num_inference_steps": gen.get("num_inference_steps"),
        "elapsed_sec": gen.get("elapsed_sec"),
        "seed": gen.get("seed"),
    })

with METRIC_EXPORT_MANIFEST_JSONL.open("w", encoding="utf-8") as f:
    for record in metric_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

fieldnames = [
    "metric_index", "experiment_id", "sample_id", "image_id", "file_name",
    "generated_image_relative_path", "coco_original_path", "background_prompt",
    "foreground_prompts", "category_names", "annotation_ids", "method",
    "model_family", "model_id", "sampler", "scheduler", "hyper_sd_repo", "hyper_sd_weight", "num_inference_steps",
    "elapsed_sec", "seed",
]
with METRIC_EXPORT_MANIFEST_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for record in metric_records:
        row = {key: record.get(key) for key in fieldnames}
        for key in ("foreground_prompts", "category_names", "annotation_ids"):
            row[key] = json.dumps(row[key], ensure_ascii=False)
        writer.writerow(row)

export_summary = {
    "experiment_id": METRIC_EXPORT_EXPERIMENT_ID,
    "num_generated_images": len(metric_records),
    "generated_images_dir": str(METRIC_EXPORT_GENERATED_DIR),
    "manifest_jsonl": str(METRIC_EXPORT_MANIFEST_JSONL),
    "manifest_csv": str(METRIC_EXPORT_MANIFEST_CSV),
    "copy_original_images": COPY_ORIGINAL_IMAGES_FOR_METRIC_EXPORT,
    "source_manifest_path": str(RUN_MANIFEST),
    "source_output_dir": str(OUTPUT_DIR),
}
with METRIC_EXPORT_SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(export_summary, f, ensure_ascii=False, indent=2)

zip_base = METRIC_EXPORT_ROOT / f"{METRIC_EXPORT_EXPERIMENT_ID}__metric_export"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=METRIC_EXPORT_DIR)
with zipfile.ZipFile(zip_path, "r") as zf:
    bad_file = zf.testzip()
if bad_file is not None:
    raise RuntimeError(f"Zip integrity check failed at: {bad_file}")

print("[OK] Metric export is ready:", METRIC_EXPORT_DIR)
print("[OK] Zip export:", zip_path)
display(pd.DataFrame(metric_records)[["metric_index", "sample_id", "image_id", "generated_image_relative_path", "elapsed_sec"]].head())


## 2. Đo Metric Sau Generation

Phần dưới dùng ảnh đã sinh trong `OUTPUT_DIR` và `generation_summary.json` để tính:

```text
FID, IS, CLIP(fg), CLIP(bg), Time(s)
```

Trước khi đo metric, notebook giải phóng model diffusion khỏi GPU để có chỗ load Inception/CLIP. Nếu muốn generate lại ảnh sau bước này, hãy chạy lại cell load `MultiDiffusionHyperSD` trước.


In [ ]:
# Giải phóng VRAM trước khi load Inception/CLIP cho metric.
import gc

for var_name in ("md_hyper", "sanity_image", "generated", "payload", "overlay", "original", "batch", "loader", "preview_batch"):
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

print("[OK] Released generation objects before metric evaluation.")


In [ ]:
# Đo FID, IS, CLIP(fg), CLIP(bg), Time(s) trên output theo RUN_PROFILE vừa generate.
from metrics import MetricEvaluationConfig, run_evaluation, write_metrics_report

generation_summary_path = OUTPUT_DIR / "generation_summary.json"
assert generation_summary_path.exists(), f"Missing generation summary: {generation_summary_path}"

metric_device = "cuda:0" if torch.cuda.is_available() else "cpu"
metric_config = MetricEvaluationConfig(
    manifest_path=RUN_MANIFEST,
    coco_root=COCO_ROOT,
    generated_dir=OUTPUT_DIR,
    generation_summary=generation_summary_path,
    output_dir=METRICS_OUTPUT_DIR,
    model_family="sd15",
    target_size=TARGET_SIZE,
    metrics=METRIC_NAMES,
    batch_size=METRIC_BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
    device=metric_device,
    clip_batch_size=CLIP_BATCH_SIZE,
    is_splits=IS_SPLITS,
)

metric_report = run_evaluation(metric_config)
metrics_json, metrics_csv = write_metrics_report(
    metric_report,
    METRICS_OUTPUT_DIR,
    prefix=METRICS_REPORT_PREFIX,
)

values = metric_report["metrics"]


def fmt(value: object, digits: int = 4) -> str:
    if value is None:
        return "-"
    try:
        value = float(value)
        if math.isnan(value):
            return "-"
        return f"{value:.{digits}f}"
    except Exception:
        return str(value)


metrics_table = pd.DataFrame([
    {"Metric": "FID↓", "Value": fmt(values.get("fid"))},
    {"Metric": "IS↑", "Value": fmt(values.get("is_mean"))},
    {"Metric": "IS std", "Value": fmt(values.get("is_std"))},
    {"Metric": "CLIP(fg)↑", "Value": fmt(values.get("clip_fg_x100"))},
    {"Metric": "CLIP(bg)↑", "Value": fmt(values.get("clip_bg_x100"))},
    {"Metric": "Time(s)↓", "Value": fmt(values.get("time_mean_sec"))},
    {"Metric": "Total time(s)", "Value": fmt(values.get("time_total_sec"))},
])

display(Markdown(
    f"## Metric Done\n"
    f"- evaluated: `{metric_report['num_evaluated']}` / `{metric_report['num_manifest_records']}` samples\n"
    f"- missing generated images: `{metric_report['num_missing_generated']}`\n"
    f"- metrics JSON: `{metrics_json}`\n"
    f"- metrics CSV: `{metrics_csv}`"
))
display(metrics_table)

metric_report
